In [1]:
import numpy as np
import joblib
import json

# load the terrain susceptibility layer
susceptibility = np.load("../data/processed/periyar_susceptibility.npy")

with open("../data/processed/periyar_grid_meta.json") as f:
    grid_meta = json.load(f)

# load the trained rainfall model
model = joblib.load("../models/rainfall_lgbm.pkl")

print("Susceptibility shape:", susceptibility.shape)
print("Grid bbox:", grid_meta["bbox"])

Susceptibility shape: (3600, 4320)
Grid bbox: [75.99986111114504, 9.5001388888822, 77.1998611111452, 10.500138888882333]


In [8]:
print("Type of susceptibility:", type(susceptibility))
print("Susceptibility shape:", susceptibility.shape)
print("Any NaNs in susceptibility:", np.isnan(susceptibility).any())
print("Susceptibility min/max (ignoring NaN):", np.nanmin(susceptibility), np.nanmax(susceptibility))

result = predict_inundation_risk(90.0, model, susceptibility)

print()
print("Type of inundation_map:", type(result["inundation_map"]))
print("Heavy rain alert:", result["heavy_rain_alert"])
print("ML risk score:", result["ml_risk_score"])
print("Combined rainfall risk:", result["rainfall_risk"])
print("Inundation map shape:", result["inundation_map"].shape)
print("Inundation map max (ignoring NaN):", np.nanmax(result["inundation_map"]))

Type of susceptibility: <class 'numpy.ndarray'>
Susceptibility shape: (3600, 4320)
Any NaNs in susceptibility: True
Susceptibility min/max (ignoring NaN): 0.0 1.0

Type of inundation_map: <class 'numpy.ndarray'>
Heavy rain alert: 1
ML risk score: 0.01401176825641781
Combined rainfall risk: 1.0
Inundation map shape: (3600, 4320)
Inundation map max (ignoring NaN): 1.0


C:\Users\prash\rainfall-warning\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [7]:
# replace NaN pixels (coastal flat-terrain artifacts) with 0 — treat "unknown flow direction" as "no elevated risk"
susceptibility_clean = np.nan_to_num(susceptibility, nan=0.0)

np.save("../data/processed/periyar_susceptibility.npy", susceptibility_clean)
print("Re-saved cleaned susceptibility array")
print("NaNs remaining:", np.isnan(susceptibility_clean).any())

Re-saved cleaned susceptibility array
NaNs remaining: False


In [9]:
result_heavy = predict_inundation_risk(90.0, model, susceptibility_clean)
result_light = predict_inundation_risk(10.0, model, susceptibility_clean)

print("Heavy rain (90mm) — risk:", result_heavy["rainfall_risk"], "map max:", result_heavy["inundation_map"].max())
print("Light rain (10mm)  — risk:", result_light["rainfall_risk"], "map max:", result_light["inundation_map"].max())

C:\Users\prash\rainfall-warning\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\prash\rainfall-warning\.venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


Heavy rain (90mm) — risk: 1.0 map max: 1.0
Light rain (10mm)  — risk: 0.0005718679248851851 map max: 0.0005718679248851851
